# 05 — Quantitative MRI in MS

Conventional T1/T2/FLAIR are *qualitative* — pixel intensity has no physical meaning. Quantitative MRI (qMRI) measures tissue properties directly, enabling cross-site comparison and microstructural insight.

Three qMRI metrics with established MS relevance:

| Metric | What it measures | MS relevance |
|---|---|---|
| **T1 mapping** | Longitudinal relaxation time | Sensitive to tissue water and myelin |
| **T2 mapping** | Transverse relaxation time | Inflammation, oedema |
| **MTR** (magnetisation transfer ratio) | Macromolecular content (myelin) | Demyelination / remyelination |

This notebook is a *teaching scaffold* — actual qMRI requires specific acquisition sequences (variable flip angle for T1, multi-echo for T2, MT-on/MT-off for MTR). Replace paths with your own qMRI data.

## MTR — example calculation

In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path

DATA = Path('/neurodesktop-storage/ms-workshop-data/sub-001/qmri')
MT_ON  = nib.load(DATA / 'sub-001_MTon.nii.gz')
MT_OFF = nib.load(DATA / 'sub-001_MToff.nii.gz')

on = MT_ON.get_fdata()
off = MT_OFF.get_fdata()
with np.errstate(divide='ignore', invalid='ignore'):
    mtr = np.where(off > 0, 100 * (off - on) / off, 0)

out_img = nib.Nifti1Image(mtr, MT_OFF.affine, MT_OFF.header)
out_path = Path('/neurodesktop-storage/ms-workshop-out/sub-001/qmri')
out_path.mkdir(parents=True, exist_ok=True)
nib.save(out_img, out_path/'MTR.nii.gz')
print('Saved MTR map. Mean MTR (in NAWM ROI) ~ 30–40 percent for healthy tissue.')

## Sample MTR within a lesion mask

Reusing the lesion mask from notebook 02 to compare lesion vs normal-appearing white matter (NAWM) MTR:

In [ ]:
lesion = nib.load('/neurodesktop-storage/ms-workshop-out/sub-001/samseg/lesions.nii.gz').get_fdata().astype(bool)
# (assume MTR has been resampled to same space as lesion mask)
mtr_lesion = mtr[lesion]
mtr_nawm   = mtr[(~lesion) & (mtr > 5)]  # crude NAWM proxy — refine in real analysis
print(f'MTR lesion: {mtr_lesion.mean():.1f} ± {mtr_lesion.std():.1f}')
print(f'MTR NAWM  : {mtr_nawm.mean():.1f}  ± {mtr_nawm.std():.1f}')

## Clinical interpretation

- Lesional MTR is typically reduced by 20–50% vs NAWM, reflecting demyelination.
- Recovery of MTR over time within a lesion suggests remyelination — a key endpoint for trials of neuroregenerative therapies.
- NAWM MTR reductions exist even in normal-appearing tissue and predict disability progression.

## Tools available in Neurodesk for advanced qMRI

- **qMRLab** (MATLAB/Octave) — comprehensive qMRI fitting (T1, T2*, MT, NODDI, qMT)
- **hMRI toolbox** (SPM-based) — multi-parameter mapping (MPM)
- **MRtrix3** — DWI-based microstructure (NODDI, FBA)